# Phân tích mô hình rủi ro giao trễ

Notebook này **đọc** kết quả của một lần huấn luyện, không huấn luyện lại. Chạy trước:

```bash
uv run python -m app.scripts.train_risk_model
```

Nguồn dữ liệu: bốn tệp trong `RISK_MODEL_DIR`, cộng tệp CSV Olist gốc khi cần soi đơn cụ thể.

Báo cáo JSON trả lời câu hỏi *bao nhiêu*; notebook này trả lời câu hỏi *vì sao* — đặc trưng nào
dẫn dắt, xác suất có được hiệu chỉnh không, và mô hình sai ở những đơn nào.

**Lưu notebook với ô kết quả đã xoá sạch.** Số liệu đã nằm ở JSON và CSV; giữ thêm một bản trong
notebook chỉ làm git diff nhiễu và tệp phình to.

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd

from app.core.config import settings
from app.risk.predictor import CHECKPOINTS, MODEL_FILENAME
from app.risk.training import (
    PREDICTIONS_FILENAME,
    REPORT_FILENAME,
    THRESHOLD_GRID,
    metrics_at,
)

MODEL_DIR = Path(settings.RISK_MODEL_DIR)

report = json.loads((MODEL_DIR / REPORT_FILENAME).read_text("utf-8"))
bundle = joblib.load(MODEL_DIR / MODEL_FILENAME)
predictions = pd.read_csv(MODEL_DIR / PREDICTIONS_FILENAME)

print(f"Phiên bản mô hình : {report['model_version']}")
print(f"Thuật toán đã chọn: {report['selected_algorithm']}")
print(f"Ngưỡng đề xuất    : {report['suggested_risk_threshold']}")
print(f"Đạt F1 >= {report['f1_target']}   : {report['meets_f1_target']}")

## 1. Ba thuật toán ở ba mốc dự đoán

Hai con số cho mỗi ô, và chúng trả lời hai câu khác nhau:

- `best_f1` — F1 tốt nhất mà phép quét ngưỡng tìm được **trên chính tập kiểm tra**. Đây là tiêu chí
  chọn thuật toán (ADR-0008), nhưng nó lạc quan: ngưỡng được chọn bằng cách nhìn vào đáp án.
- `at_selected_threshold` — F1 khi dùng ngưỡng lấy từ tập kiểm định. Đây mới là con số sẽ nhận
  được khi triển khai.

In [ ]:
rows = []
for algorithm, evaluation in report["algorithms"].items():
    for checkpoint in CHECKPOINTS:
        entry = evaluation["test"][checkpoint]
        chosen = entry["at_selected_threshold"]
        rows.append(
            {
                "thuật toán": algorithm,
                "mốc": checkpoint,
                "F1 (ngưỡng thật)": chosen["f1"],
                "precision": chosen["precision"],
                "recall": chosen["recall"],
                "F1 tốt nhất (quét)": entry["best_f1"],
            }
        )

pd.DataFrame(rows).pivot(index="thuật toán", columns="mốc")

## 2. Ba tập chia và tỷ lệ trễ của chúng

Đây là ô quan trọng nhất để đọc đúng mọi con số phía trên. Dữ liệu chia **theo thời điểm đặt hàng**,
và tỷ lệ trễ của Olist tụt mạnh theo thời gian. F1 trên tập kiểm tra vì thế không so được với F1 đo
trên dữ liệu trộn ngẫu nhiên — trộn ngẫu nhiên cho điểm đẹp hơn nhưng là điểm giả, vì mô hình thật
luôn dự đoán cho đơn đặt sau mọi đơn nó đã học.

In [ ]:
splits = pd.DataFrame(report["splits"]).T
splits["late_rate_%"] = (splits["late_rate"].astype(float) * 100).round(2)
display(splits)

print("Đơn bị loại khỏi dữ liệu huấn luyện:")
for reason, count in report["excluded_orders"].items():
    print(f"  {reason:<28} {count:>7,}")

## 3. Đặc trưng nào dẫn dắt từng chặng

Ba chặng có ba bộ đặc trưng quan trọng khác nhau, và đó chính là lý do ADR-0008 tách chặng thay vì
dự đoán một nhãn trễ chung. Nếu `distance_km` không nổi lên ở `carrier_transit`, hoặc
`seller_prior_*` không nổi lên ở `seller_handling`, thì có gì đó sai ở tầng đặc trưng.

In [ ]:
# Đọc qua API chung của StageModel thay vì thọc vào thuộc tính riêng của từng thư viện:
# candidates.py hứa bên gọi không cần biết bên dưới là XGBoost hay Scikit-learn.
features = report["features"]

for stage, model in bundle.stage_models.items():
    values = model.feature_importances()
    if values is None:
        print(f"{stage}: thuật toán này không phơi độ quan trọng đặc trưng")
        continue
    series = pd.Series(values, index=features).sort_values().tail(12)
    series.plot.barh(figsize=(7, 4), title=f"Đặc trưng quan trọng nhất — {stage}")
    plt.tight_layout()
    plt.show()


## 4. Xác suất trễ có được hiệu chỉnh không

Trục ngang là xác suất mô hình nói ra, trục dọc là tỷ lệ trễ thật trong nhóm đơn đó. Đường chéo là
hiệu chỉnh hoàn hảo. Đường nằm **dưới** đường chéo nghĩa là mô hình nói quá — chuyện thường gặp khi
xác suất suy ra từ mô phỏng chứ không được huấn luyện trực tiếp trên nhãn trễ.

Hiệu chỉnh kém không làm F1 hỏng, vì ngưỡng vốn được dò theo dữ liệu chứ không đặt cứng ở 0,5.

In [ ]:
fig, axes = plt.subplots(1, len(CHECKPOINTS), figsize=(14, 4), sharey=True)

for axis, checkpoint in zip(axes, CHECKPOINTS):
    part = predictions[predictions["checkpoint"] == checkpoint]
    bins = pd.qcut(part["late_probability"], 10, duplicates="drop")
    grouped = part.groupby(bins, observed=True).agg(
        predicted=("late_probability", "mean"), actual=("is_late", "mean")
    )
    axis.plot(grouped["predicted"], grouped["actual"], marker="o")
    limit = float(max(grouped.max().max(), 0.05))
    axis.plot([0, limit], [0, limit], linestyle="--", linewidth=1)
    axis.set_title(checkpoint)
    axis.set_xlabel("xác suất dự đoán")

axes[0].set_ylabel("tỷ lệ trễ thật")
plt.tight_layout()
plt.show()

## 5. F1, precision và recall theo ngưỡng

Đường dựng lại phép quét mà lệnh huấn luyện đã chạy. Vạch dọc là ngưỡng đề xuất.

Nếu đỉnh F1 nằm gần 0,5 thì có gì đó sai: tỷ lệ trễ nền chỉ khoảng 6,8%, nên ở mốc đặt hàng gần như
không đơn nào đạt xác suất 0,5.

In [ ]:
threshold = report["suggested_risk_threshold"]
fig, axes = plt.subplots(1, len(CHECKPOINTS), figsize=(14, 4), sharey=True)

for axis, checkpoint in zip(axes, CHECKPOINTS):
    part = predictions[predictions["checkpoint"] == checkpoint]
    probability = part["late_probability"].to_numpy()
    truth = part["is_late"].to_numpy(dtype=bool)
    curve = pd.DataFrame(
        [metrics_at(probability, truth, value) for value in THRESHOLD_GRID]
    ).set_index("threshold")
    curve.plot(ax=axis, legend=(checkpoint == CHECKPOINTS[0]))
    axis.axvline(threshold, color="grey", linestyle="--", linewidth=1)
    axis.set_title(checkpoint)
    axis.set_xlabel("ngưỡng")

plt.tight_layout()
plt.show()

## 6. Mô hình sai ở những đơn nào

Hai loại sai có giá khác hẳn nhau trong nghiệp vụ:

- **Bỏ sót** (đơn trễ mà xác suất thấp) — nhân viên không được cảnh báo, không can thiệp gì.
- **Báo động nhầm** (đơn đúng hạn mà xác suất cao) — tốn công can thiệp vô ích, và nhiều quá thì
  người dùng bắt đầu bỏ qua cảnh báo.

Ở mốc đặt hàng, ngưỡng thấp đổi rất nhiều báo động nhầm lấy việc bắt được thêm đơn trễ. Bảng dưới
cho biết cái giá đó cụ thể là bao nhiêu đơn.

In [ ]:
at_order = predictions[predictions["checkpoint"] == "order_placed"].copy()
at_order["flagged"] = at_order["late_probability"] >= threshold

matrix = pd.crosstab(
    at_order["is_late"].map({True: "thật sự trễ", False: "đúng hạn"}),
    at_order["flagged"].map({True: "bị gắn cờ", False: "không gắn cờ"}),
)
display(matrix)

missed = at_order[at_order["is_late"] & ~at_order["flagged"]]
print(f"Bỏ sót {len(missed):,} đơn trễ. Những đơn mô hình tự tin nhất mà vẫn sai:")
display(missed.nsmallest(10, "late_probability")[["order_id", "late_probability"]])